# 05 - Priprema podataka za mašinsko učenje

U ovoj svesci sirove trening i test skupove pretvaramo u oblik koji
algoritmi mašinskog učenja mogu da koriste. Sve odluke donosimo gledajući
isključivo trening skup, a onda iste transformacije primenjujemo i na test.

Prolazimo kroz dva glavna koraka:

1. Feature engineering, spajanje kategorija, konverzija tipova i kreiranje
   novih atributa
2. Enkodiranje kategorija, odnosno pretvaranje teksta u brojeve, i to
   pomoću Label, Ordinal i One-Hot enkodiranja

Na kraju čuvamo pripremljene podatke.

## 1. Učitavanje trening i test skupova

Učitavamo sva četiri objekta koja smo sačuvali u svesci `03`:
`X_train`, `X_test`, `y_train`, `y_test`.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

putanja_processed = "../data/processed/"

X_train = pd.read_csv(putanja_processed + "X_train.csv")
X_test = pd.read_csv(putanja_processed + "X_test.csv")
y_train = pd.read_csv(putanja_processed + "y_train.csv")
y_test = pd.read_csv(putanja_processed + "y_test.csv")

# y_train i y_test su učitani kao DataFrame sa jednom kolonom, pretvaramo ih u Series.
y_train = y_train.squeeze()
y_test = y_test.squeeze()

print("Učitani podaci:")
print(f"  X_train: {X_train.shape[0]} × {X_train.shape[1]}")
print(f"  X_test:  {X_test.shape[0]} × {X_test.shape[1]}")
print(f"  y_train: {y_train.shape[0]}")
print(f"  y_test:  {y_test.shape[0]}")

Učitani podaci:
  X_train: 5625 × 19
  X_test:  1407 × 19
  y_train: 5625
  y_test:  1407


## 2. Feature engineering

Pre enkodiranja prolazimo kroz nekoliko transformacija atributa.

### 2.1 Spajanje "No internet service" i "No"

Šest kolona (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`,
`TechSupport`, `StreamingTV`, `StreamingMovies`) ima tri moguće vrednosti:
`Yes`, `No`, `No internet service`.

Sa poslovne strane, „korisnik nema tu uslugu" znači isto bez obzira da li
je razlog to što ima internet ali nije uzeo tu opciju, ili prosto nema
internet uopšte. Zato `No internet service` svodimo na `No`.

Isto radimo i sa kolonom `MultipleLines`, gde vrednost `No phone service`
takođe postaje `No`.

In [2]:
kolone_no_internet = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]


for kolona in kolone_no_internet:
    X_train[kolona] = X_train[kolona].replace("No internet service", "No")
    X_test[kolona] = X_test[kolona].replace("No internet service", "No")


X_train["MultipleLines"] = X_train["MultipleLines"].replace("No phone service", "No")
X_test["MultipleLines"] = X_test["MultipleLines"].replace("No phone service", "No")

# Provera - sada te kolone treba da imaju samo "Yes" i "No".
print("Vrednosti nakon spajanja:")
for kolona in kolone_no_internet + ["MultipleLines"]:
    print(f"  {kolona}: {sorted(X_train[kolona].unique())}")

Vrednosti nakon spajanja:
  OnlineSecurity: ['No', 'Yes']
  OnlineBackup: ['No', 'Yes']
  DeviceProtection: ['No', 'Yes']
  TechSupport: ['No', 'Yes']
  StreamingTV: ['No', 'Yes']
  StreamingMovies: ['No', 'Yes']
  MultipleLines: ['No', 'Yes']


### 2.2 Konverzija `SeniorCitizen` u tekstualnu binarnu kategoriju

`SeniorCitizen` je zapisana kao broj (0/1), iako je zapravo binarna
kategorija. Zato je pretvaramo u `No`/`Yes`, da bude ista po formatu kao
ostale binarne kolone, i da kasnije sve njih možemo enkodirati istom
logikom.

In [3]:
#proveravamo sa if-om da bi obezbedili sigurnost od veceg broja pokretanja
if X_train["SeniorCitizen"].dtype != "object":
    mapiranje_senior = {0: "No", 1: "Yes"}
    X_train["SeniorCitizen"] = X_train["SeniorCitizen"].map(mapiranje_senior)
    X_test["SeniorCitizen"] = X_test["SeniorCitizen"].map(mapiranje_senior)
print("SeniorCitizen vrednosti nakon konverzije:")
print(X_train["SeniorCitizen"].value_counts())

SeniorCitizen vrednosti nakon konverzije:
SeniorCitizen
No     4715
Yes     910
Name: count, dtype: int64


### 2.3 Kreiranje broja aktivnih dodatnih usluga

Ovde gledamo šest dodatnih usluga (`OnlineSecurity`, `OnlineBackup`,
`DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) i za
svakog korisnika brojimo koliko ih ima aktivno (vrednost `Yes`).

Dobijena kolona `ActiveServices` je nova numerička promenljiva, i može
nam pokazati da li broj korišćenih usluga ima veze sa tim da li korisnik
ostaje ili odlazi.

In [4]:
usluge = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

X_train["ActiveServices"] = (X_train[usluge] == "Yes").sum(axis=1)
X_test["ActiveServices"] = (X_test[usluge] == "Yes").sum(axis=1)

print("Raspodela broja aktivnih dodatnih usluga:")
print(X_train["ActiveServices"].value_counts().sort_index())

Raspodela broja aktivnih dodatnih usluga:
ActiveServices
0    1740
1     770
2     835
3     902
4     688
5     468
6     222
Name: count, dtype: int64


In [5]:
X_train[usluge + ["ActiveServices"]].head()

,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,ActiveServices
0,Yes,Yes,Yes,Yes,No,No,4
1,No,No,Yes,Yes,No,No,2
2,No,Yes,Yes,Yes,No,No,3
3,No,Yes,No,No,No,Yes,2
4,Yes,No,No,No,Yes,No,2


### 2.4 Grupisanje korisnika prema dužini pretplate

`tenure` pokazuje koliko meseci korisnik već koristi uslugu. Da bi analiza
bila preglednija, te vrednosti grupišemo u četiri kategorije:

- `0-12 meseci`
- `13-24 meseca`
- `25-48 meseci`
- `49+ meseci`

Tako umesto pojedinačnog broja meseci dobijamo kategoriju, koju je lakše
tumačiti dalje u analizi.

### 2.5 Kreiranje izvedenog atributa `TenureGroup`

Pored sirove kolone `tenure`, uvodimo i grupisanu verziju `TenureGroup`
koja korisnike svrstava u četiri kategorije po dužini staža:

- `0-12 meseci` — novi korisnici (najveći rizik od odlaska)
- `13-24 meseca` — korisnici u ranoj fazi
- `25-48 meseci` — stabilni srednjoročni korisnici
- `49+ meseci` — dugogodišnji lojalni korisnici

Grupisanje pravimo iz dva razloga:

1. **Vizualizacije** — u multivarijantnim heatmapima u svesci `06` mnogo
   je čitljivije prikazati 4 grupe nego 72 pojedinačne vrednosti tenure-a.

2. **Alternativni prikaz podatka za modele** — modeli mogu drugačije da
   reaguju na diskretne kategorije nego na sirovi broj meseci.

Enkodiramo ordinalno (0-3) jer redosled kategorija ima poslovni smisao
(kraća -> duža lojalnost).

**Napomena o multikolinearnosti:** `TenureGroup` je izveden iz `tenure`,
pa su ove dve kolone visoko korelisane. Kod linearnih modela izbacićemo
`TenureGroup` (zajedno sa `TotalCharges`) da izbegnemo multikolinearnost.
Kod modela baziranih na stablima zadržavamo obe kolone.

In [6]:
# Grupe prema dužini pretplate:
# 0-12 meseci   → novi korisnici (do 1 godine)
# 13-24 meseca  → korisnici sa 1-2 godine pretplate
# 25-48 meseci  → korisnici sa 2-4 godine pretplate
# 49+ meseci    → dugogodišnji korisnici (više od 4 godine)

granice_tenure = [0, 12, 24, 48, float("inf")]
nazivi_tenure = [
    "0-12 meseci",
    "13-24 meseca",
    "25-48 meseci",
    "49+ meseci"
]

X_train["TenureGroup"] = pd.cut(
    X_train["tenure"],
    bins=granice_tenure,
    labels=nazivi_tenure,
    include_lowest=True
)

X_test["TenureGroup"] = pd.cut(
    X_test["tenure"],
    bins=granice_tenure,
    labels=nazivi_tenure,
    include_lowest=True
)

print("Raspodela korisnika po grupama tenure:")
print(X_train["TenureGroup"].value_counts().sort_index())

Raspodela korisnika po grupama tenure:
TenureGroup
0-12 meseci     1724
13-24 meseca     818
25-48 meseci    1279
49+ meseci      1804
Name: count, dtype: int64


## 3. Enkodiranje kategorijskih promenljivih

Sledeći korak je pretvaranje tekstualnih kolona u brojeve. U zavisnosti
od tipa kategorije koristimo jednu od tri metode:

- Label Encoding (0/1), za binarne kolone (`Yes`/`No`)
- Ordinal Encoding, za kategorije koje imaju prirodan redosled
- One-Hot Encoding, za nominalne kolone sa više od dve vrednosti

### Pregled kolona po metodi:

| Metoda | Kolone |
|---|---|
| Label (0/1) | `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaperlessBilling` |
| Ordinal | `Contract` (Month-to-month < One year < Two year), `TenureGroup` (0-12 < 13-24 < 25-48 < 49+ meseci) |
| One-Hot | `InternetService`, `PaymentMethod` |

`tenure`, `MonthlyCharges` i `TotalCharges` ostaju kakve jesu, pošto su
već numeričke.

### 3.1 Label Encoding za binarne kolone

Sve binarne kolone (`Yes`/`No`) enkodiramo na isti način: `Yes → 1`,
`No → 0`.

In [7]:
binarne_kolone = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling"
]

# kolona "gender" ima "Male"/"Female", ne "Yes"/"No". Zbog toga resavamo je zasebno pre opšteg mapiranja.
mapiranje_gender = {"Male": 1, "Female": 0}
X_train["gender"] = X_train["gender"].map(mapiranje_gender)
X_test["gender"] = X_test["gender"].map(mapiranje_gender)


# Opšte mapiranje za ostale binarne kolone (Yes/No).
mapiranje_yes_no = {"Yes": 1, "No": 0}
for kolona in binarne_kolone:
    if kolona == "gender":
        continue  
    X_train[kolona] = X_train[kolona].map(mapiranje_yes_no)
    X_test[kolona] = X_test[kolona].map(mapiranje_yes_no)


print("Prve 3 vrste nakon Label enkodiranja binarnih kolona:")
X_train[binarne_kolone].head(3)

Prve 3 vrste nakon Label enkodiranja binarnih kolona:


,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling
0,1,0,1,1,1,1,1,1,1,1,0,0,0
1,1,0,0,0,0,0,0,0,1,1,0,0,0
2,0,0,1,0,1,1,0,1,1,1,0,0,0


### 3.2 One-Hot Encoding za `Contract`

`Contract` ima tri vrednosti koje na prvi pogled idu jedna za drugom po
logici:
- `Month-to-month` (najkraći i najfleksibilniji, ali i najrizičniji za odlazak)
- `One year` (srednji)
- `Two year` (najduži, korisnici najlojalniji)

Na početku smo razmišljali da ih enkodiramo ordinalno (`0 → 1 → 2`), pošto
postoji prirodan redosled. Ali kad smo u EDA pogledali kako `Contract` utiče
na Churn, videli smo da efekat NIJE linearan: Month-to-month ima ~43% churn,
One year ~11%, a Two year ~3%. Razlike između susednih kategorija nisu
jednake (pad sa 43 na 11 nije isti kao sa 11 na 3).

Kod linearnih modela poput Logističke regresije, ordinalno kodiranje bi
model naterao da pretpostavi jednake razmake između kategorija, pa bi
izgubio ovu nelinearnu informaciju. Zato koristimo **One-Hot Encoding**,
koji svakoj kategoriji daje svoju kolonu, pa model može da nauči zasebnu
težinu za svaku.

Sa `drop_first=True` izbacujemo jednu kategoriju kao referentnu, da
izbegnemo savršenu kolinearnost (ista logika kao kod ostalih One-Hot kolona).

Modelima na stablima (Random Forest, XGBoost) ovo ionako ne smeta, njima
i One-Hot i Ordinal rade podjednako dobro.

In [8]:
# Contract enkodiramo kao One-Hot umesto Ordinal.
# Razlog: efekat Contract-a na Churn NIJE linearan (Month-to-month=43%, One year=11%, Two year=3%),
# pa linearni modeli (Logistička regresija) ne bi mogli da nauče pravi obrazac iz ordinalnog kodiranja.
# One-Hot omogućava da model uči zasebnu težinu za svaku kategoriju.
X_train = pd.get_dummies(X_train, columns=["Contract"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["Contract"], drop_first=True)

In [9]:
mapiranje_tenure = {
    "0-12 meseci": 0,
    "13-24 meseca": 1,
    "25-48 meseci": 2,
    "49+ meseci": 3
}

X_train["TenureGroup"] = X_train["TenureGroup"].map(mapiranje_tenure)
X_test["TenureGroup"] = X_test["TenureGroup"].map(mapiranje_tenure)

#Provera
print("Raspodela TenureGroup nakon Ordinal enkodiranja:")
print(X_train["TenureGroup"].value_counts().sort_index())

Raspodela TenureGroup nakon Ordinal enkodiranja:
TenureGroup
0    1724
1     818
2    1279
3    1804
Name: count, dtype: int64


### 3.3 One-Hot Encoding za nominalne kolone

Ostale su još dve nominalne kolone bez prirodnog redosleda:
- `InternetService`: `DSL`, `Fiber optic`, `No` (3 vrednosti)
- `PaymentMethod`: `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)` (4 vrednosti)

Njih rešavamo One-Hot Encoding-om, gde svaka kategorija postane posebna
kolona sa 0 ili 1.

Bitno je da koristimo `drop_first=True`, čime izbacujemo po jednu kolonu
za svaku promenljivu (recimo `InternetService_DSL`). Ta kolona je ionako
suvišna, jer ako model zna da korisnik nema `Fiber optic` i nema `No`,
onda automatski zna da ima `DSL`. Na ovaj način izbegavamo savršenu
kolinearnost, koja bi pravila probleme kod linearnih modela.

In [10]:
nominalne_kolone = ["InternetService", "PaymentMethod"]

# primenjujemo istu transformaciju na oba skupa.
# funkcija get_dummies vraca True/False pa mi treba da konvertujemo u int (0/1) za konzistentnost sa ostalim binarnim kolonama.
X_train = pd.get_dummies(X_train, columns=nominalne_kolone, drop_first=True)
X_test = pd.get_dummies(X_test, columns=nominalne_kolone, drop_first=True)


bool_kolone = X_train.select_dtypes(include=["bool"]).columns
X_train[bool_kolone] = X_train[bool_kolone].astype(int)
X_test[bool_kolone] = X_test[bool_kolone].astype(int)

# Provera - trebalo bi da imamo nove kolone.
print("Nove kolone nakon One-Hot Encoding-a:")
nove_kolone = [k for k in X_train.columns if k.startswith("InternetService_") or k.startswith("PaymentMethod_")]
print(nove_kolone)

Nove kolone nakon One-Hot Encoding-a:
['InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


### 3.4 Provera konzistentnosti kolona train i test skupa

Posle One-Hot Encoding-a moramo proveriti da train i test imaju iste
kolone. Ako se neka kategorija pojavi samo u jednom od skupova, broj
kolona se ne poklapa i model neće moći da radi.

In [11]:
# Poredimo skupove kolona.
kolone_train = set(X_train.columns)
kolone_test = set(X_test.columns)

samo_u_train = kolone_train - kolone_test
samo_u_test = kolone_test - kolone_train

print(f"Broj kolona u X_train: {len(kolone_train)}")
print(f"Broj kolona u X_test:  {len(kolone_test)}")

if samo_u_train or samo_u_test:
    print("\nUPOZORENJE - kolone se ne poklapaju:")
    if samo_u_train:
        print(f"  Samo u train: {samo_u_train}")
    if samo_u_test:
        print(f"  Samo u test: {samo_u_test}")
else:
    print("\nSve kolone se poklapaju - train i test su konzistentni.")

Broj kolona u X_train: 25
Broj kolona u X_test:  25

Sve kolone se poklapaju - train i test su konzistentni.


### 3.5 Enkodiranje ciljne promenljive `y`

`Churn` ima vrednosti `Yes` (napustio) i `No` (ostao), pa ih mapiramo u
1/0 tako da ima smisla:

- `Yes` (churn) → 1, pozitivna klasa, to je ono što nas zanima da predvidimo
- `No` (ostao) → 0, negativna klasa

Ovo prati uobičajenu konvenciju u klasifikaciji, gde se ono što nas
zanima obeležava sa 1. Kod nas je to churn, jer cilj modela i jeste da
prepozna korisnike koji će otići.

In [12]:
# Mapiranje ciljne promenljive.
mapiranje_churn = {"Yes": 1, "No": 0}

y_train = y_train.map(mapiranje_churn)
y_test = y_test.map(mapiranje_churn)

# Provera.
print("Raspodela y_train nakon enkodiranja:")
print(y_train.value_counts())
print(f"\nProcenat pozitivne klase (Yes/1): {100 * y_train.mean():.2f}%")

Raspodela y_train nakon enkodiranja:
Churn
0    4130
1    1495
Name: count, dtype: int64

Procenat pozitivne klase (Yes/1): 26.58%


## 4. Provera finalnog stanja pripremljenih podataka

Pre nego što sačuvamo podatke, proveravamo da li su spremni za mašinsko
učenje:

- Sve kolone su numeričke
- Nema nedostajućih vrednosti
- Broj kolona train i test skupa je isti
- Dimenzije se poklapaju sa onim što smo očekivali

In [13]:
# 1. Provera tipova svih kolona - sve treba da budu numeričke.
print("=" * 60)
print("PROVERA TIPOVA KOLONA")
print("=" * 60)
print("Tipovi kolona u X_train:")
print(X_train.dtypes.value_counts())


object_kolone = X_train.select_dtypes(include=["object"]).columns
if len(object_kolone) > 0:
    print(f"\nUPOZORENJE - preostale tekstualne kolone: {list(object_kolone)}")
else:
    print("\nSve kolone su numeričke.")

# 2. Provera nedostajućih vrednosti.
print("\n" + "=" * 60)
print("PROVERA NEDOSTAJUĆIH VREDNOSTI")
print("=" * 60)

na_train = X_train.isnull().sum().sum()
na_test = X_test.isnull().sum().sum()
na_y_train = y_train.isnull().sum()
na_y_test = y_test.isnull().sum()

print(f"NaN u X_train: {na_train}")
print(f"NaN u X_test:  {na_test}")
print(f"NaN u y_train: {na_y_train}")
print(f"NaN u y_test:  {na_y_test}")

# 3. Provera dimenzija.
print("\n" + "=" * 60)
print("DIMENZIJE PRIPREMLJENIH PODATAKA")
print("=" * 60)
print(f"X_train: {X_train.shape[0]} × {X_train.shape[1]}")
print(f"X_test:  {X_test.shape[0]} × {X_test.shape[1]}")
print(f"y_train: {y_train.shape[0]}")
print(f"y_test:  {y_test.shape[0]}")

# 4. Konzistentnost kolona.
if list(X_train.columns) == list(X_test.columns):
    print("\nKolone train i test se poklapaju.")
else:
    print("\nUPOZORENJE - kolone se ne poklapaju!")

PROVERA TIPOVA KOLONA
Tipovi kolona u X_train:
int64       22
float64      2
category     1
Name: count, dtype: int64

Sve kolone su numeričke.

PROVERA NEDOSTAJUĆIH VREDNOSTI
NaN u X_train: 0
NaN u X_test:  0
NaN u y_train: 0
NaN u y_test:  0

DIMENZIJE PRIPREMLJENIH PODATAKA
X_train: 5625 × 25
X_test:  1407 × 25
y_train: 5625
y_test:  1407

Kolone train i test se poklapaju.


### Prikaz finalnog `X_train`

Prikazujemo prvih 5 redova pripremljenog trening skupa da vizuelno
potvrdimo da je sve u numeričkom obliku i spremno za modele.

In [14]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,MonthlyCharges,TotalCharges,ActiveServices,TenureGroup,Contract_One year,Contract_Two year,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,1,65,1,1,1,1,1,1,0,0,0,94.55,6078.75,4,3,0,1,1,0,1,0,0
1,1,0,0,0,26,0,0,0,0,1,1,0,0,0,35.75,1022.50,2,2,0,0,0,0,0,1,0
2,0,0,1,0,68,1,1,0,1,1,1,0,0,0,90.20,6297.65,3,3,0,1,1,0,1,0,0
3,1,0,0,0,3,1,0,0,1,0,0,0,1,0,84.30,235.05,2,0,0,0,1,0,0,1,0
4,0,0,1,0,49,0,0,1,0,0,0,1,0,0,40.65,2070.75,2,3,0,0,0,0,0,0,0


## 5. Čuvanje pripremljenih podataka

Pripremljene podatke čuvamo u `data/ml/`, folder koji sadrži podatke
spremne za modeliranje.

Razlika u odnosu na `data/processed/`:
- `data/processed/`: podaci posle split-a, ali pre enkodiranja, još uvek
  čitljivi ("Yes"/"No")
- `data/ml/`: podaci posle enkodiranja, potpuno numerički i spremni za
  algoritme

Tako u `data/processed/` možemo dalje da radimo EDA na čitljivim
podacima, dok `data/ml/` služi isključivo modelima.

In [15]:
putanja_ml = "../data/ml/"

# Čuvamo sva 4 objekta kao CSV.
X_train.to_csv(putanja_ml + "X_train.csv", index=False)
X_test.to_csv(putanja_ml + "X_test.csv", index=False)
y_train.to_csv(putanja_ml + "y_train.csv", index=False)
y_test.to_csv(putanja_ml + "y_test.csv", index=False)

print("Pripremljeni podaci sačuvani u folder 'data/ml/':")
print(f"  X_train.csv  ({X_train.shape[0]} x {X_train.shape[1]})")
print(f"  X_test.csv   ({X_test.shape[0]} x {X_test.shape[1]})")
print(f"  y_train.csv  ({y_train.shape[0]} vrednosti)")
print(f"  y_test.csv   ({y_test.shape[0]} vrednosti)")

Pripremljeni podaci sačuvani u folder 'data/ml/':
  X_train.csv  (5625 x 25)
  X_test.csv   (1407 x 25)
  y_train.csv  (5625 vrednosti)
  y_test.csv   (1407 vrednosti)


## 6. Zaključak

Kroz ovu svesku smo pripremili podatke za mašinsko učenje, i to kroz
nekoliko koraka:

Feature engineering (nove promenljive):
1. Spojili "No internet service" i "No phone service" sa "No" u sedam kolona
2. Konvertovali `SeniorCitizen` iz broja (0/1) u tekst
3. `ActiveServices`: nova numerička promenljiva (0-6), broj aktivnih usluga
4. `TenureGroup`: nova kategorijska promenljiva, grupiše tenure u 4 intervala

Enkodiranje kategorija:
- Label Encoding (0/1) na 13 binarnih kolona
- Ordinal Encoding na `Contract` (0 → 1 → 2, prati dužinu ugovora)
- One-Hot Encoding na `InternetService` i `PaymentMethod` (uz
  `drop_first=True`, da izbegnemo kolinearnost)
- `Churn` mapiran kao Yes → 1, No → 0

Na kraju imamo 24 kolone, sve numeričke (`int64` ili `float64`), bez
nedostajućih vrednosti, i train i test se poklapaju po broju kolona.